## Import libraries

In [1]:
# %%
# Prototype: CNN + Grad-CAM on CBIS-DDSM subset

import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import models
from torchvision import transforms

# Make project root importable
ROOT_DIR = os.path.abspath(os.path.join(".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from pathlib import Path

#from utils.dataset import CBISDicomDataset  # you'll create/adjust this
from utils.dataset import CBISDataset

#from utils.transforms import get_preprocessing_transforms
from utils.gradcam import GradCAM

from torchvision import transforms, models
from torch.utils.data import DataLoader, random_split

from sklearn.metrics import roc_auc_score, f1_score
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

## Point to archive

In [2]:
RAW_ROOT = os.path.join(ROOT_DIR, "data", "raw", "archive")

CSV_DIR = os.path.join(RAW_ROOT, "csv")
JPEG_DIR = os.path.join(RAW_ROOT, "jpeg")

meta_path = os.path.join(CSV_DIR, "meta.csv")
dicom_info_path = os.path.join(CSV_DIR, "dicom_info.csv")

In [3]:
meta = pd.read_csv(meta_path)
#meta.head()

In [4]:
dicom_info = pd.read_csv(dicom_info_path)
#dicom_info.head()

In [5]:
# check image types in dataset
dicom_info.SeriesDescription.unique()

array(['cropped images', 'full mammogram images', nan, 'ROI mask images'],
      dtype=object)

In [6]:
# check image path in dataset
# cropped images
cropped_images = dicom_info[dicom_info.SeriesDescription=='cropped images'].image_path
#cropped_images.head(5)

In [7]:
#full mammogram images
full_mammo = dicom_info[dicom_info.SeriesDescription=='full mammogram images'].image_path
#full_mammo.head(5)

In [8]:
# ROI images
roi_img = dicom_info[dicom_info.SeriesDescription=='ROI mask images'].image_path
#roi_img.head(5)

In [9]:
# change directory path of images
cropped_images = cropped_images.replace('CBIS-DDSM/jpeg', JPEG_DIR, regex=True)
full_mammo = full_mammo.replace('CBIS-DDSM/jpeg', JPEG_DIR, regex=True)
roi_img = roi_img.replace('CBIS-DDSM/jpeg', JPEG_DIR, regex=True)

# view new paths
#print('Cropped Images paths:\n')
#print(cropped_images.iloc[0])
#print('Full mammo Images paths:\n')
#print(full_mammo.iloc[0])
#print('ROI Mask Images paths:\n')
#print(roi_img.iloc[0])

In [10]:
# convert into dictionaries keyed by the UID folder name
from pathlib import Path

# Build dictionaries: UID → full JPEG path
full_mammo_dict = {
    Path(p).parts[-2]: p
    for p in full_mammo
}

cropped_images_dict = {
    Path(p).parts[-2]: p
    for p in cropped_images
}

roi_img_dict = {
    Path(p).parts[-2]: p
    for p in roi_img
}

In [11]:
# load the mass dataset
mass_train = pd.read_csv(os.path.join(CSV_DIR, "mass_case_description_train_set.csv"))
mass_test = pd.read_csv(os.path.join(CSV_DIR, "mass_case_description_test_set.csv"))

#mass_train.head()

In [12]:
#print(len(cropped_images), len(roi_img), len(mass_train))

## Convert to arrays for training

In [13]:
# --- Clean column names globally ---
#mass_train.columns = mass_train.columns.str.strip()
#mass_test.columns = mass_test.columns.str.strip()

# --- Rename columns consistently ---
mass_train = mass_train.rename(columns={
    "image file path": "img_path",
    "cropped image file path": "cropped_path",
    "ROI mask file path": "mask_path"
})
mass_test = mass_test.rename(columns={
    "image file path": "img_path",
    "cropped image file path": "cropped_path",
    "ROI mask file path": "mask_path"
})


In [14]:
def fix_image_path(df):
    """
    Replace DICOM-style relative paths in mass_train/mass_test
    with correct JPEG paths using lookup dictionaries.
    """
    for idx, row in df.iterrows():

        # --- Fix full mammogram path ---
        uid2 = row["img_path"].split("/")[2]  # second UID folder
        if uid2 in full_mammo_dict:
            df.at[idx, "img_path"] = full_mammo_dict[uid2]

        # --- Fix cropped path ---
        uid2 = row["cropped_path"].split("/")[2]
        if uid2 in cropped_images_dict:
            df.at[idx, "cropped_path"] = cropped_images_dict[uid2]

        # --- Fix ROI mask path ---
        uid2 = row["mask_path"].split("/")[2]
        if uid2 in roi_img_dict:
            df.at[idx, "mask_path"] = roi_img_dict[uid2]

fix_image_path(mass_train)
fix_image_path(mass_test)

In [15]:
# --- Combine train + test ---
full_mass = pd.concat([mass_train, mass_test], axis=0).reset_index(drop=True)

# --- Strip spaces again after concat ---
full_mass.columns = full_mass.columns.str.strip()

# --- Safety check ---
#print("Columns:", full_mass.columns.tolist())

# --- Convert relative paths to absolute JPEG paths ---
def safe_join(base_dir, path):
    if isinstance(path, str):
        return os.path.join(base_dir, path)
    return None

#print(full_mass.columns.tolist())
#full_mass

In [16]:
#print(full_mass.columns.tolist())
#print(full_mass)

#full_mass["img_path"] = full_mass["img_path"].apply(lambda x: safe_join(JPEG_DIR, x))
#full_mass["cropped_path"] = full_mass["cropped_path"].apply(lambda x: safe_join(JPEG_DIR, x))
#full_mass["mask_path"] = full_mass["mask_path"].apply(lambda x: safe_join(JPEG_DIR, x))

#full_mass["img_path"] = JPEG_DIR + "/" + full_mass["img_path"].astype(str)
#full_mass["cropped_path"] = JPEG_DIR + "/" + full_mass["cropped_path"].astype(str)
#full_mass["mask_path"] = JPEG_DIR + "/" + full_mass["mask_path"].astype(str)

for col in ["img_path", "cropped_path", "mask_path"]:
    full_mass[col] = full_mass[col].astype(str).map(lambda p: str(Path(JPEG_DIR) / p))


# --- Create label ---
class_mapper = {"MALIGNANT": 1, "BENIGN": 0, "BENIGN_WITHOUT_CALLBACK": 0}
full_mass["label"] = full_mass["pathology"].replace(class_mapper)

# --- Use cropped images for training ---
#full_mass["img_path"] = full_mass["cropped_path"]
full_mass.loc[:, "img_path"] = full_mass["cropped_path"]


# --- Filter valid paths ---
#full_mass = full_mass[
#    full_mass["img_path"].apply(lambda x: isinstance(x, str) and os.path.exists(x))
#]

#full_mass = full_mass[
#    full_mass["mask_path"].apply(lambda y: isinstance(y, str) and os.path.exists(y))
#]

#full_mass = full_mass.reset_index(drop=True)

# Vectorized existence check
img_exists = full_mass["img_path"].astype(str).map(os.path.exists)
mask_exists = full_mass["mask_path"].astype(str).map(os.path.exists)

p = full_mass.loc[0, "img_path"]
#print("Path:", p)
#print("Exists:", os.path.exists(p))



#print(img_exists.value_counts())
#print(mask_exists.value_counts())

# Filter in one step
full_mass = full_mass[img_exists & mask_exists].reset_index(drop=True)

#print(full_mass.columns.tolist())
#print(full_mass)

In [17]:
# --- Final training DataFrame ---
#print(full_mass.columns.tolist())
train_df = full_mass[["img_path", "mask_path", "label"]]
#print(train_df.head())
train_df = train_df.sample(n=200, random_state=42).reset_index(drop=True)

In [18]:
IMG_SIZE = 512

train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((512, 512)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                     std=[0.229, 0.224, 0.225]),
    transforms.ToTensor(),
])


val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                     std=[0.229, 0.224, 0.225]),
])


full_dataset = CBISDataset(train_df, transform=train_tf)

val_ratio = 0.2
val_size = int(len(full_dataset) * val_ratio)
train_size = len(full_dataset) - val_size

train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
val_ds.dataset.transform = val_tf

#train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
#val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)

In [19]:
imgs, masks, labels = next(iter(train_loader))
#print("imgs:", imgs.shape)
#print("masks:", masks.shape)
#print("labels:", labels.shape)

## Model, optimizer, scheduler, early stopping, checkpointing

In [20]:
from torchvision import models
import torch.nn as nn
import torch.optim as optim

resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
num_features = resnet.fc.in_features

#for param in resnet.layer1.parameters():
#    param.requires_grad = False
#for param in resnet.layer2.parameters():
#    param.requires_grad = False
for name, param in resnet.named_parameters():
    if "layer3" not in name and "layer4" not in name and "fc" not in name:
        param.requires_grad = False


#resnet.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(num_features, 1))
resnet.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_features, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 1)
)

resnet = resnet.to(device)

#criterion = nn.BCEWithLogitsLoss()
pos_weight = torch.tensor([3.0]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

#optimizer = optim.Adam(resnet.parameters(), lr=1e-4, weight_decay=1e-4)
optimizer = torch.optim.Adam([
    {"params": resnet.layer3.parameters(), "lr": 1e-5},
    {"params": resnet.layer4.parameters(), "lr": 1e-5},
    {"params": resnet.fc.parameters(), "lr": 1e-4},
], weight_decay=1e-5)

#scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, verbose=True
)


/usr/local/lib/python3.8/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


In [21]:
#print(train_df["mask_path"].head())
#print(train_df["mask_path"].apply(type).value_counts())

## Training loop (prototype: ~5–10 epochs)

In [22]:
# %%
# Training loop (prototype: ~5–10 epochs)

#from tqdm.auto import tqdm

#scaler = torch.cuda.amp.GradScaler()

# def train_one_epoch(model, loader, optimizer, criterion, device):
#     model.train()
#     running_loss = 0.0

#     for imgs, masks, labels in loader:
#         imgs = imgs.to(device)
#         labels = labels.float().to(device)

#         optimizer.zero_grad()

#         outputs = model(imgs).squeeze(1)
#         loss = criterion(outputs, labels)

#         loss.backward()

#         # Gradient clipping
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)

#         optimizer.step()

#         running_loss += loss.item() * imgs.size(0)

#     return running_loss / len(loader.dataset)


# def evaluate(model, loader, criterion, device):
#     model.eval()
#     running_loss = 0.0
#     all_outputs = []
#     all_labels = []

#     with torch.no_grad():
#         for imgs, masks, labels in tqdm(loader, desc="Val", leave=False):
#             imgs = imgs.to(device)
#             labels = labels.float().to(device)

#             outputs = model(imgs).squeeze(1)
#             loss = criterion(outputs, labels)

#             running_loss += loss.item() * imgs.size(0)
#             all_outputs.append(outputs.cpu())
#             all_labels.append(labels.cpu())

#     all_outputs = torch.cat(all_outputs)
#     all_labels = torch.cat(all_labels)

#     probs = torch.sigmoid(all_outputs)
#     preds = (probs > 0.5).int()
#     acc = (preds == all_labels.int()).float().mean().item()
#     auc = roc_auc_score(all_labels, probs)
#     f1 = f1_score(all_labels, preds)

#     return running_loss / len(loader.dataset), acc, auc, f1

best_val_loss = float("inf")
patience = 3
wait = 0
EPOCHS = 10

#for epoch in range(EPOCHS):
#    train_loss = train_one_epoch(resnet, train_loader, optimizer, criterion, device)
#    val_loss, val_acc, val_auc, val_f1 = evaluate(resnet, val_loader, criterion, device)
#    scheduler.step(val_loss)
#    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.3f}| Val AUC: {val_auc:.3f} | Val F1: {val_f1:.3f}")

# --- Training loop ---
for epoch in range(EPOCHS):
    # Train
    resnet.train()
    running_loss = 0.0
    for imgs, masks, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        imgs = imgs.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()
        outputs = resnet(imgs).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(resnet.parameters(), max_norm=2.0)
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # Evaluate
    resnet.eval()
    val_loss, all_outputs, all_labels = 0.0, [], []
    with torch.no_grad():
        for imgs, masks, labels in val_loader:
            imgs = imgs.to(device)
            labels = labels.float().to(device)
            outputs = resnet(imgs).squeeze(1)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            all_outputs.append(outputs.cpu())
            all_labels.append(labels.cpu())

    val_loss /= len(val_loader.dataset)
    all_outputs = torch.cat(all_outputs)
    all_labels = torch.cat(all_labels)
    probs = torch.sigmoid(all_outputs)
    preds = (probs > 0.5).int()
    val_acc = (preds == all_labels.int()).float().mean().item()
    val_auc = roc_auc_score(all_labels, probs)
    val_f1 = f1_score(all_labels, preds)

    # Scheduler + early stopping
    scheduler.step(val_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        wait = 0
        torch.save(resnet.state_dict(), "best_model.pt")
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping triggered.")
            break

    # Log metrics
    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.3f} | Val AUC: {val_auc:.3f} | Val F1: {val_f1:.3f}")

Epoch 1/10:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1/10 | Train Loss: 1.3158 | Val Loss: 1.4514 | Val Acc: 0.600 | Val AUC: 0.719 | Val F1: 0.704


Epoch 2/10:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 2/10 | Train Loss: 1.1927 | Val Loss: 1.3696 | Val Acc: 0.575 | Val AUC: 0.721 | Val F1: 0.721


Epoch 3/10:   0%|          | 0/10 [00:00<?, ?it/s]

...

Epoch 6/10 | Train Loss: 1.0040 | Val Loss: 1.2146 | Val Acc: 0.650 | Val AUC: 0.744 | Val F1: 0.720


Epoch 7/10:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 7/10 | Train Loss: 1.0176 | Val Loss: 1.2727 | Val Acc: 0.650 | Val AUC: 0.714 | Val F1: 0.696


Epoch 8/10:   0%|          | 0/10 [00:00<?, ?it/s]

Early stopping triggered.
